# Phase 5a: RAGAS evaluation

Stream A: faithfulness and answer relevancy for baseline vs hybrid-RAG (`phase4b` JSONL outputs).

**Pipeline:** `phase4b` → this notebook → `phase5b_llm_judge_eval.ipynb` → `phase5c_pca_kmeans.R` → `phase5d_cross_stream_eval.ipynb`.

**Outputs:** `data/06_evaluation/ragas_*_v14.csv` and `data/06_evaluation/figures/5a_ragas/v14/*.png`.

#Composer through a Cursor IDE guided the development of this script


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datasets import Dataset
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import answer_relevancy, faithfulness
from ragas.run_config import RunConfig

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 140
load_dotenv()


In [ ]:
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "scripts" else Path.cwd().resolve()
SCRIPT_DIR = BASE_DIR / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
from pipeline_constants import (
    BASELINE_JSONL,
    HYBRID_JSONL,
    INFERENCE_TAG,
    RAGAS_EVAL_TAG,
)

BASELINE_INPUT = BASELINE_JSONL
HYBRID_INPUT = HYBRID_JSONL
OUTPUT_DIR = BASE_DIR / "data" / "06_evaluation"
TAG = RAGAS_EVAL_TAG
PROMPT_VERSION = INFERENCE_TAG
JUDGE_MODEL = "gpt-4o-mini"
RUN_RAGAS = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
hybrid_csv = OUTPUT_DIR / f"ragas_hybrid_scores_{TAG}.csv"
baseline_csv = OUTPUT_DIR / f"ragas_baseline_scores_{TAG}.csv"

assert BASELINE_INPUT.exists(), f"Missing: {BASELINE_INPUT}"
assert HYBRID_INPUT.exists(), f"Missing: {HYBRID_INPUT}"
assert INFERENCE_TAG in BASELINE_INPUT.name and INFERENCE_TAG in HYBRID_INPUT.name, (
    f"Expected {INFERENCE_TAG} JSONL paths; got baseline={BASELINE_INPUT.name} hybrid={HYBRID_INPUT.name}"
)

print(BASELINE_INPUT)
print(HYBRID_INPUT)
print(f"TAG={TAG} PROMPT_VERSION={PROMPT_VERSION} RUN_RAGAS={RUN_RAGAS}")
print(f"Output CSVs: {baseline_csv.name}, {hybrid_csv.name}")


## 1. Helpers, preflight, and dataset formatting

Run before setting `RUN_RAGAS = True`. Validates JSONL shape, id alignment, answer extraction, and `OPENAI_API_KEY` (no RAGAS API calls).


In [ ]:
def extract_answer_only(text: str) -> str:
    if not isinstance(text, str):
        return ""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    for ln in lines:
        if ln.lower().startswith("answer:"):
            return ln.split(":", 1)[1].strip()
    return text.strip()


def load_and_format_data(filepath, context_mapping=None):
    questions, answers, contexts, ids, categories = [], [], [], [], []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            ids.append(data["id"])
            categories.append(data.get("category", "Unknown"))
            questions.append(data["prompt"])
            answers.append(extract_answer_only(data["ai_response"]))
            if context_mapping is not None:
                contexts.append([context_mapping.get(data["id"], "")])
            else:
                contexts.append([data.get("retrieved_context", "")])
    return Dataset.from_dict({
        "id": ids,
        "category": categories,
        "question": questions,
        "answer": answers,
        "contexts": contexts,
    })


def run_preflight_checks() -> None:
    import os

    load_dotenv()
    api_key = (os.getenv("OPENAI_API_KEY") or "").strip()
    if not api_key or api_key.startswith("your-"):
        raise ValueError("OPENAI_API_KEY missing or placeholder in .env")

    def load_ids(path: Path) -> list[int]:
        ids: list[int] = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    ids.append(int(json.loads(line)["id"]))
        return ids

    base_ids = load_ids(BASELINE_INPUT)
    hyb_ids = load_ids(HYBRID_INPUT)
    if len(base_ids) != 250 or len(hyb_ids) != 250:
        raise ValueError(f"Expected 250 rows each; got baseline={len(base_ids)} hybrid={len(hyb_ids)}")
    if base_ids != hyb_ids:
        raise ValueError("Baseline and hybrid id order differ")

    empty_ctx = 0
    missing_answer = 0
    with HYBRID_INPUT.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            ctx = row.get("retrieved_context", "") or ""
            if not ctx.strip() or ctx.strip() == "NONE - BASELINE":
                empty_ctx += 1

    with BASELINE_INPUT.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if not extract_answer_only(row.get("ai_response", "")):
                missing_answer += 1

    cached = baseline_csv.exists() and hybrid_csv.exists()
    print(
        f"Preflight OK | tag={TAG} rows=250 | empty_ctx={empty_ctx} missing_answers={missing_answer} | "
        f"RUN_RAGAS={RUN_RAGAS} cached_csvs={cached}"
    )


run_preflight_checks()


## 2. Run RAGAS (or load cached CSVs)

Set `RUN_RAGAS = True` to recompute scores. Requires `OPENAI_API_KEY` in `.env`.


In [ ]:
if RUN_RAGAS:
    context_map = {}
    with open(HYBRID_INPUT, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            context_map[row["id"]] = row.get("retrieved_context", "")

    hybrid_dataset = load_and_format_data(HYBRID_INPUT)
    baseline_dataset = load_and_format_data(BASELINE_INPUT, context_mapping=context_map)
    print(f"Datasets: hybrid={len(hybrid_dataset)} baseline={len(baseline_dataset)}")

    judge_llm = ChatOpenAI(model=JUDGE_MODEL, temperature=0.0)
    judge_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    run_config = RunConfig(max_workers=2, max_retries=20, max_wait=90)
    metrics_to_run = [faithfulness, answer_relevancy]

    print("Evaluating hybrid-RAG...")
    hybrid_results = evaluate(
        dataset=hybrid_dataset,
        metrics=metrics_to_run,
        llm=judge_llm,
        embeddings=judge_embeddings,
        run_config=run_config,
        raise_exceptions=False,
    )
    print("Evaluating baseline...")
    baseline_results = evaluate(
        dataset=baseline_dataset,
        metrics=metrics_to_run,
        llm=judge_llm,
        embeddings=judge_embeddings,
        run_config=run_config,
        raise_exceptions=False,
    )
    hybrid_df = hybrid_results.to_pandas()
    baseline_df = baseline_results.to_pandas()
    hybrid_df.to_csv(hybrid_csv, index=False)
    baseline_df.to_csv(baseline_csv, index=False)
    print(f"Saved: {hybrid_csv}")
    print(f"Saved: {baseline_csv}")
else:
    if not hybrid_csv.exists() or not baseline_csv.exists():
        raise FileNotFoundError(
            f"Missing RAGAS CSVs. Set RUN_RAGAS=True or run evaluation first: {baseline_csv}, {hybrid_csv}"
        )
    hybrid_df = pd.read_csv(hybrid_csv)
    baseline_df = pd.read_csv(baseline_csv)
    print(f"Loaded cached RAGAS CSVs: {baseline_csv.name}, {hybrid_csv.name}")


## 3. Summary


In [ ]:
summary = pd.DataFrame([
    {
        "dataset": "baseline",
        "faithfulness": baseline_df["faithfulness"].mean(),
        "answer_relevancy": baseline_df["answer_relevancy"].mean(),
    },
    {
        "dataset": "hybrid_rag",
        "faithfulness": hybrid_df["faithfulness"].mean(),
        "answer_relevancy": hybrid_df["answer_relevancy"].mean(),
    },
])
summary


## 4. Figures


In [ ]:
def _prepare_plot_df(baseline_df: pd.DataFrame, hybrid_df: pd.DataFrame) -> pd.DataFrame:
    b = baseline_df.copy()
    h = hybrid_df.copy()
    b["dataset"] = "baseline"
    h["dataset"] = "hybrid_rag"
    plot_df = pd.concat([b, h], ignore_index=True)

    keep = [c for c in ["id", "category", "dataset", "faithfulness", "answer_relevancy"] if c in plot_df.columns]
    plot_df = plot_df[keep].copy()

    for col in ["faithfulness", "answer_relevancy"]:
        if col in plot_df.columns:
            plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

    return plot_df


def plot_overall_metric_bars(plot_df: pd.DataFrame, fig_dir: Path):
    metrics = ["faithfulness", "answer_relevancy"]
    mdf = (
        plot_df.melt(id_vars=["dataset"], value_vars=metrics, var_name="metric", value_name="score")
        .dropna(subset=["score"])
    )

    plt.figure(figsize=(8, 5))
    ax = sns.barplot(data=mdf, x="metric", y="score", hue="dataset", errorbar=None)
    ax.set_title("RAGAS Mean Scores by Dataset")
    ax.set_xlabel("")
    ax.set_ylabel("Score")
    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%.3f",
            label_type="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
    plt.tight_layout()
    plt.savefig(fig_dir / "5a_overall_metric_means.png", bbox_inches="tight")
    plt.close()


def plot_category_metric_bars(plot_df: pd.DataFrame, fig_dir: Path):
    metrics = ["faithfulness", "answer_relevancy"]
    mdf = (
        plot_df.melt(id_vars=["category", "dataset"], value_vars=metrics, var_name="metric", value_name="score")
        .dropna(subset=["score"])
    )

    g = sns.catplot(
        data=mdf,
        kind="bar",
        x="category",
        y="score",
        hue="dataset",
        col="metric",
        errorbar=None,
        height=5,
        aspect=1.4,
        sharey=True,
    )
    g.set_xticklabels(rotation=30, ha="right")
    g.fig.suptitle("RAGAS Scores by Category (Stratified)", y=1.03)
    g.savefig(fig_dir / "5a_category_metric_bars.png", bbox_inches="tight")
    plt.close("all")


def plot_category_delta_heatmap(plot_df: pd.DataFrame, fig_dir: Path):
    means = (
        plot_df.groupby(["category", "dataset"], as_index=False)[["faithfulness", "answer_relevancy"]]
        .mean(numeric_only=True)
    )

    metrics = ["faithfulness", "answer_relevancy"]
    pivot_b = means[means["dataset"] == "baseline"].set_index("category")[metrics]
    pivot_h = means[means["dataset"] == "hybrid_rag"].set_index("category")[metrics]
    delta = (pivot_h - pivot_b).sort_index()
    overall = plot_df.groupby("dataset")[metrics].mean(numeric_only=True)
    delta.loc["Overall Delta"] = overall.loc["hybrid_rag"] - overall.loc["baseline"]

    fig, ax = plt.subplots(figsize=(7.5, 5.4))
    sns.heatmap(
        delta,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        center=0,
        cbar_kws={"label": "Hybrid - Baseline"},
        ax=ax,
    )
    ax.set_title("Category-Level RAGAS Delta")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Category")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(fig_dir / "5a_category_delta_heatmap.png", bbox_inches="tight")
    plt.close()


def save_ragas_figures(plot_df: pd.DataFrame, fig_dir: Path):
    if plot_df.empty:
        print("No plot data available for 5a figures.")
        return
    plot_df = plot_df.copy()
    plot_df["category"] = plot_df.get("category", "Unknown").fillna("Unknown")
    plot_overall_metric_bars(plot_df, fig_dir)
    plot_category_metric_bars(plot_df, fig_dir)
    plot_category_delta_heatmap(plot_df, fig_dir)
    print(f"5a figures saved to: {fig_dir}")


In [ ]:
fig_dir = OUTPUT_DIR / "figures" / "5a_ragas" / TAG
fig_dir.mkdir(parents=True, exist_ok=True)

plot_df = _prepare_plot_df(baseline_df, hybrid_df)
save_ragas_figures(plot_df, fig_dir)
